# CreditLens AI, milestones 1 and 2: synthetic data and the data pipeline

Runs end to end on the free Colab tier. No GPU needed for this notebook.

1. Generate every synthetic dataset from one seed.
2. Look at how messy the raw bank statements are.
3. Run the pipeline: raw to clean to curated, with Pandera contracts.
4. Prove the cleaned data matches the truth exactly.

## Setup
Replace the repo URL with your own GitHub repo.

In [ ]:
!apt-get -qq install -y poppler-utils > /dev/null
!git clone https://github.com/YOUR-USERNAME/creditlens-ai.git
%cd creditlens-ai
!pip install -q -e ".[dev]"

## 1. Generate the data
Same seed, same data, every time.

In [ ]:
!python -m creditlens.generators

## 2. Look at the raw mess
Predict before you run: how many different date formats will you find in one file?

In [ ]:
import pandas as pd
raw = pd.read_csv("data/raw/statements/A002_statement.csv", dtype=str, keep_default_na=False)
raw.head(15)

In [ ]:
import json
issues = json.load(open("data/answer_key/statement_issues.json"))  # instructor only
pd.DataFrame(issues).T.head(10)

## 3. Run the pipeline
Each wrangling step lives in `src/creditlens/pipeline/wrangle.py` as a small, tested function.

In [ ]:
!python -m creditlens.pipeline --load

In [ ]:
quality = pd.read_csv("data/clean/quality_report.csv")
quality.sort_values("duplicates_removed", ascending=False).head(10)

In [ ]:
features = pd.read_csv("data/curated/statement_features.csv")
features.sort_values("cheque_returns_12m", ascending=False).head(10)

## 4. Prove it
The test suite compares every cleaned transaction with the generator's truth.

In [ ]:
!pytest -q

## 5. Day 1 lab: concurrency
Predict which strategy wins, then run it.

In [ ]:
!python -m creditlens.pipeline.ingest